# Implementation of the combination of the HO-CAE and FSS Algoirthms on the 2.45 GHz Spectrum
This notebook combines the fss.ipynb and the ho-cae.ipynb implementations of their respective algoirthms, and uses them to process and find low power bands on an active 2.45g spectrum. This code will be build out and applied live in gnuradio as the next test, as well as the further research of the algoirthms using deep-learning

## Pre-Processing
import, validate, and plot data to ensure everything is good

In [ ]:
from utils.data import load_iq_data

FILE_NAME = "usrp_2.45g_test_capture.bin"

# Load the IQ data from the binary file
data = load_iq_data(FILE_NAME)
print(f"Loaded {len(data)} samples from {FILE_NAME}")

In [ ]:
# Get all of the constants of the data. 
# In the future we should store this as metadata alongside the .bin file, but for now we will hardcode it here.
SAMP_RATE = 100e6
CENTER_FREQ = 2.45e9
DURATION = len(data) / SAMP_RATE


In [ ]:
from utils.plot import plot_spectrogram

# plot a section of the data to understand what it looks like before the algorithm
data_to_plot = data[:int(SAMP_RATE * 0.01)]  # Plot the first 0.01 seconds of data
plot_spectrogram(data_to_plot, SAMP_RATE, CENTER_FREQ)

In [ ]:
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

# Ensure IQ data is a one-dimensional complex array
data = np.asarray(data).squeeze()

if data.ndim != 1:
    raise ValueError(f"Expected 1D IQ data, received shape {data.shape}")

# STFT parameters
NFFT = 1024
OVERLAP = 512

# Compute the short-time Fourier transform
f, t, z = signal.stft(
    data,
    fs=SAMP_RATE,
    window="hann",
    nperseg=NFFT,
    noverlap=OVERLAP,
    nfft=NFFT,
    return_onesided=False,
    boundary=None,
    padded=False,
)

# Put negative frequencies on the left and positive frequencies on the right
f_shifted = np.fft.fftshift(f)
z_shifted = np.fft.fftshift(z, axes=0)

magnitude_sq = np.abs(z_shifted) ** 2
magnitude_sq = magnitude_sq.T

magnitude_db = 20.0 * np.log10(np.maximum(np.abs(z_shifted), 1e-12))
magnitude_db = magnitude_db.T

print(f"magnitude_sq max value: {np.max(magnitude_sq)}, min value: {np.min(magnitude_sq)}, mean value: {np.mean(magnitude_sq)}")

## Import the Algoirthms and Apply them to the Spectral Data
Then we plot to see the algorthm working. 

In [ ]:
import sys
sys.path.append("../algorithms")

from fss import FSSStateMachine
from hocae import HO_CAE

hocae = HO_CAE(n=64, k=5, alpha=16)
fss = FSSStateMachine()

In [ ]:
bin_data = []
for time_step in magnitude_sq:
    threshold = hocae.compute_threshold(time_step)
    fss.threshold = threshold

    for freq_bin in time_step:
        fss.cycle(freq_bin)

    temp = fss.RESET()
    bin_data.append(temp)   

In [ ]:
# plot the spectrogram with the detected buckets overlaid
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

magnitude_db = np.asarray(magnitude_db)
f_shifted = np.asarray(f_shifted)
t = np.asarray(t)

print("magnitude_db:", magnitude_db.shape)
print("frequency bins:", len(f_shifted))
print("time bins:", len(t))

# Force C into shape:
# rows = time
# columns = frequency
if magnitude_db.shape == (len(f_shifted), len(t)):
    plot_data = magnitude_db.T
elif magnitude_db.shape == (len(t), len(f_shifted)):
    plot_data = magnitude_db
else:
    raise ValueError(
        f"Unexpected spectrogram shape {magnitude_db.shape}. "
        f"Expected {(len(f_shifted), len(t))} or "
        f"{(len(t), len(f_shifted))}."
    )

fig, ax = plt.subplots(figsize=(12, 24))

spectrogram = ax.pcolormesh(
    f_shifted,
    t,
    plot_data,
    shading="auto",
    cmap="gray",
)

df = np.median(np.diff(f_shifted))
dt = np.median(np.diff(t))

for time_idx, (start_bin, bucket_size) in enumerate(bin_data):
    if time_idx >= len(t):
        break

    start_bin = int(start_bin)
    bucket_size = int(bucket_size)

    if bucket_size <= 0:
        continue

    end_bin = start_bin + bucket_size - 1

    start_bin = np.clip(start_bin, 0, len(f_shifted) - 1)
    end_bin = np.clip(end_bin, 0, len(f_shifted) - 1)

    start_frequency = f_shifted[start_bin]
    end_frequency = f_shifted[end_bin] + df

    rectangle = Rectangle(
        (
            start_frequency,
            t[time_idx] - dt / 2,
        ),
        width=end_frequency - start_frequency,
        height=dt,
        facecolor="red",
        edgecolor="none",
        alpha=0.15,
    )

    ax.add_patch(rectangle)

ax.set_title("Spectrogram with Detected Buckets")
ax.set_xlabel("Frequency Offset [Hz]")
ax.set_ylabel("Time [s]")

# fig.colorbar(
#     spectrogram,
#     ax=ax,
#     label="Intensity [dB]",
# )

plt.tight_layout()
plt.show()